<h1> Final </h1>

In [1]:
import subprocess
import os
import time
import pandas as pd  # Added: Required for slimming logic
import shutil        # Added: Required for cleaning up temp folders
from pathlib import Path

# --- CONFIG ---
LOCAL_DATA_DIR = Path(r"D:\English")
LOCAL_RESULT_DIR = Path(r"D:\English_Results")
LOCAL_RESULT_DIR.mkdir(parents=True, exist_ok=True)

REMOTE_USER_IP = "ubuntu@129.146.121.121"
REMOTE_INBOX_DIR = "/home/ubuntu/hist-llm-data3/input_data/"
REMOTE_OUTBOX_PATH = "/home/ubuntu/hist-llm-data3/output_data/"
SSH_KEY = r"C:\Users\danielyoon\Documents\lambda_nanochat_openssh"

def verify_parquet_file(filepath: Path) -> bool:
    """Verify parquet file is complete (has magic bytes at start and end)."""
    if not filepath.exists():
        return False
    try:
        with open(filepath, 'rb') as f:
            if f.read(4) != b'PAR1':
                return False
            f.seek(-4, 2)
            if f.read(4) != b'PAR1':
                print(f"    [WARN] Missing end magic bytes (truncated)")
                return False
        return True
    except Exception as e:
        print(f"    [WARN] Verification error: {e}")
        return False

def slim_year_data(source_dir, slim_dir, max_chars=3000):
    """Truncates text to ~512 tokens locally to reduce upload payload."""
    slim_dir.mkdir(parents=True, exist_ok=True)
    for p_file in source_dir.glob("*.parquet"):
        df = pd.read_parquet(p_file)
        # BGE-large v1.5 truncates at 512 tokens; 3k chars is a safe upper bound
        df['text'] = df['text'].astype(str).str[:max_chars] 
        df.to_parquet(slim_dir / p_file.name)

def run_workflow():
    years = sorted([d.name for d in LOCAL_DATA_DIR.iterdir() if d.is_dir() and d.name.isdigit()])
    years = [y for y in years if int(y) >= 2023] 
    
    for year in years:
        if (LOCAL_RESULT_DIR / f"embeddings_{year}.parquet").exists():
            print(f"Year {year} already processed. Skipping...")
            continue

        year_dir = LOCAL_DATA_DIR / year
        slim_temp_dir = Path(r"D:\temp_slim") / year 
        archive_path = Path(r"D:\temp") / f"year_{year}.tar"
        
        # Step 1: SLIM & ARCHIVE
        print(f"\n>>> Step 1: Slimming and Archiving Year {year}...")
        slim_year_data(year_dir, slim_temp_dir) 
        subprocess.run(["tar", "-cvf", str(archive_path), "-C", str(slim_temp_dir), "."], check=True)

        # Step 2: SECURE UPLOAD
        archive_name = f"year_{year}.tar"
        temp_remote_path = f"{REMOTE_INBOX_DIR}{archive_name}.tmp"
        final_remote_path = f"{REMOTE_INBOX_DIR}{archive_name}"
        
        print(f">>> Step 2: Uploading slimmed {archive_name} safely...")
        subprocess.run(["scp", "-i", SSH_KEY, str(archive_path), f"{REMOTE_USER_IP}:{temp_remote_path}"], check=True)
        subprocess.run(["ssh", "-i", SSH_KEY, REMOTE_USER_IP, f"mv {temp_remote_path} {final_remote_path}"], check=True)
        
        # Cleanup local temp files immediately to save SSD space
        os.remove(archive_path)
        shutil.rmtree(slim_temp_dir)

        # Step 3: WAIT FOR RESULTS
        print(f">>> Step 3: Waiting for Lambda to finish embedding {year}...")
        remote_result_file = f"{REMOTE_OUTBOX_PATH}embeddings_{year}.parquet"
        while True:
            check_cmd = ["ssh", "-i", SSH_KEY, REMOTE_USER_IP, f"test -f {remote_result_file} && echo 'exists'"]
            result = subprocess.run(check_cmd, capture_output=True, text=True)
            if "exists" in result.stdout:
                print(f">>> Found results for {year}!")
                break
            time.sleep(30) 

        # Step 4: PULL & VERIFY BEFORE CLEANUP
        print(f">>> Step 4: Downloading results...")
        local_file = LOCAL_RESULT_DIR / f"embeddings_{year}.parquet"
        subprocess.run(["scp", "-i", SSH_KEY, f"{REMOTE_USER_IP}:{remote_result_file}", str(LOCAL_RESULT_DIR)], check=True)
        
        # Verify before deleting remote (only checks magic bytes, no size filter)
        if verify_parquet_file(local_file):
            print(f"    [OK] File verified ({local_file.stat().st_size / 1024 / 1024:.1f} MB). Deleting remote...")
            subprocess.run(["ssh", "-i", SSH_KEY, REMOTE_USER_IP, f"rm {remote_result_file}"], check=True)
            print(f">>> Year {year} cycle complete.")
        else:
            print(f"    [FAIL] File corrupted! Keeping remote, deleting local.")
            if local_file.exists():
                local_file.unlink()
            print(f">>> Year {year} FAILED - will retry on next run.")

if __name__ == "__main__":
    run_workflow()


>>> Step 1: Slimming and Archiving Year 2023...
>>> Step 2: Uploading slimmed year_2023.tar safely...
>>> Step 3: Waiting for Lambda to finish embedding 2023...
>>> Found results for 2023!
>>> Step 4: Downloading results...
    [WARN] Missing end magic bytes (truncated)
    [FAIL] File corrupted! Keeping remote, deleting local.
>>> Year 2023 FAILED - will retry on next run.


<h1> Sanity Check </h1>

In [33]:
year = "1701"
archive_path = LOCAL_DATA_DIR / "temp" / f"year_{year}.tar"

In [34]:
archive_path

WindowsPath('D:/English/temp/year_1701.tar')

In [6]:
import pandas as pd
check = pd.read_parquet("D:\embeddings\embeddings_1701.parquet")

<>:2: SyntaxWarning: invalid escape sequence '\e'
<>:2: SyntaxWarning: invalid escape sequence '\e'
C:\Users\danielyoon\AppData\Local\Temp\ipykernel_47876\2074325535.py:2: SyntaxWarning: invalid escape sequence '\e'
  check = pd.read_parquet("D:\embeddings\embeddings_1701.parquet")


In [8]:
check["row_idx"].unique()

array([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16,
       17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33,
       34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50,
       51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67,
       68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84,
       85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99],
      dtype=int64)

In [10]:
check[check["subset_source"] == "subset_37.parquet"]

,original_index,subset_source,row_idx,embedding
0,commentaryonepis0005epgo_13,subset_37.parquet,0,"[-5.994964e-05, -0.02866076, 0.03536555, -0.00..."
1,bim_eighteenth-century_the-great-historical-ge...,subset_37.parquet,1,"[0.0028889778, -0.046099614, 0.042529434, 0.00..."
2,bim_eighteenth-century_the-whole-works-of-the-...,subset_37.parquet,2,"[0.022117395, -0.029930104, 0.027423447, 0.030..."
3,bim_eighteenth-century_the-merchant-citizen-an...,subset_37.parquet,3,"[-0.00024121892, -0.035392325, 0.06359001, 0.0..."
4,bim_eighteenth-century_the-design-of-christian...,subset_37.parquet,4,"[0.028062802, -0.02590621, 0.05601986, -0.0166..."
...,...,...,...,...
76,bim_eighteenth-century_the-right-honourable-th...,subset_37.parquet,76,"[0.039139483, -0.028642062, 0.034983072, 0.037..."
77,bim_eighteenth-century_an-exposition-of-the-cr...,subset_37.parquet,77,"[0.03214076, -0.05143677, 0.03200041, -0.01611..."
78,bim_eighteenth-century_a-discourse-of-the-spec...,subset_37.parquet,78,"[-0.0022060678, -0.04761847, 0.064364076, 0.00..."
79,bim_eighteenth-century_serious-exhortations-to...,subset_37.parquet,79,"[0.0015664541, -0.02044756, 0.049699806, 0.004..."


In [12]:
check2 = pd.read_parquet(r"D:\English\1701\subset_37.parquet")